# Exploratory Data Analysis (EDA) & Trends

In this notebook we will explore our cleaned e-commerce dataset, identify key monthly trends, seasonal spikes, and overall performance metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# Try to load the processed data, fallback to raw if not exists
try:
    df = pd.read_csv('../data/processed/cleaned_transactions.csv')
    print("Loaded cleaned data.")
except FileNotFoundError:
    print("Cleaned data not found. Loading raw data instead.")
    df = pd.read_csv('../data/raw/transactions.csv')
    
df['date'] = pd.to_datetime(df['date'])
df.head()

## 1. Monthly Revenue Trends & Seasonal Spikes
Let's see how revenue changes over the months to spot any seasonal trends.

In [ ]:
# Create month-year column for grouping if not exists
if 'year_month' not in df.columns:
    df['year_month'] = df['date'].dt.to_period('M').astype(str)

monthly_sales = df.groupby('year_month')['revenue'].sum().reset_index()

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_sales, x='year_month', y='revenue', marker="o", linewidth=2.5, color="#1f77b4")
plt.title('Monthly Revenue Trend', fontsize=18, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Total Revenue ($)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/figures/monthly_revenue_trend.png', dpi=300)
plt.show()

## 2. Top-10 Product Ranking
Which products are driving the most revenue for the business?

In [ ]:
top_products = df.groupby('product_id')['revenue'].sum().nlargest(10).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=top_products, x='revenue', y='product_id', palette="viridis")
plt.title('Top 10 Products by Revenue', fontsize=16, fontweight='bold')
plt.xlabel('Total Revenue ($)', fontsize=12)
plt.ylabel('Product ID', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/top_10_products.png', dpi=300)
plt.show()

## 3. Regional Profit Analysis
Understanding which regions are the most profitable can guide marketing and shipping strategies.

In [ ]:
regional_profit = df.groupby('region')['profit'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=regional_profit, x='region', y='profit', palette="magma")
plt.title('Total Profit by Region', fontsize=16, fontweight='bold')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Total Profit ($)', fontsize=12)

# Add value labels on top of bars
for index, row in regional_profit.iterrows():
    plt.text(index, row.profit, f"${row.profit:,.0f}", color='black', ha="center", va="bottom")
    
plt.tight_layout()
plt.savefig('../outputs/figures/regional_profit.png', dpi=300)
plt.show()

## 4. Category Performance Matrix
Let's look at a scatter plot to analyze revenue vs profit density across different categories.

In [ ]:
cat_performance = df.groupby('category').agg({
    'revenue': 'sum',
    'profit': 'sum',
    'quantity': 'sum'
}).reset_index()

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=cat_performance, 
    x="revenue", 
    y="profit", 
    hue="category", 
    size="quantity", 
    sizes=(100, 1000), 
    alpha=0.7,
    palette="Set2"
)
plt.title('Category Performance: Revenue vs Profit (Size = Total Quantity Sold)', fontsize=16, fontweight='bold')
plt.xlabel('Total Revenue ($)', fontsize=12)
plt.ylabel('Total Profit ($)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/figures/category_performance.png', dpi=300)
plt.show()